# Patrón de Comportamiento: Observer

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Observer** define una relación **uno a muchos**: cuando un objeto (el
**sujeto**) cambia de estado, **notifica automáticamente** a todos sus **observadores**
suscritos, sin acoplarse a sus clases concretas.

### ¿Qué problema resuelve en la banca?
Cuando ocurre un **movimiento** en una cuenta (depósito o retiro), hay que **notificar**
por varios canales: **SMS**, **email** y la **app** móvil. Si la cuenta llama
directamente a cada canal, queda acoplada a todos ellos y hay que modificarla cada vez que
se agrega o quita un canal. Observer permite **suscribir/desuscribir** canales sin tocar
la cuenta.

## Código *sin patrón* (el problema es evidente)
La cuenta invoca a mano cada canal de notificación dentro de sus métodos.

In [ ]:
class CuentaSinPatron:
    def __init__(self, saldo: float):
        self.saldo = saldo

    def registrar_movimiento(self, monto: float):
        self.saldo += monto
        # La cuenta conoce y llama a mano a CADA canal
        print(f"   [SMS]   Movimiento de ${monto}. Saldo: ${self.saldo}")
        print(f"   [EMAIL] Movimiento de ${monto}. Saldo: ${self.saldo}")
        print(f"   [APP]   Movimiento de ${monto}. Saldo: ${self.saldo}")


c = CuentaSinPatron(100000)
c.registrar_movimiento(50000)
c.registrar_movimiento(-20000)
print(">> Problema: la cuenta esta acoplada a SMS/EMAIL/APP; agregar o quitar un canal la obliga a cambiar.")

   [SMS]   Movimiento de $50000. Saldo: $150000
   [EMAIL] Movimiento de $50000. Saldo: $150000
   [APP]   Movimiento de $50000. Saldo: $150000
   [SMS]   Movimiento de $-20000. Saldo: $130000
   [EMAIL] Movimiento de $-20000. Saldo: $130000
   [APP]   Movimiento de $-20000. Saldo: $130000
>> Problema: la cuenta esta acoplada a SMS/EMAIL/APP; agregar o quitar un canal la obliga a cambiar.


### Análisis del problema
- La cuenta **conoce** cada canal concreto y los llama directamente.
- Agregar (push, webhook) o quitar un canal obliga a **modificar** la cuenta.
- No se pueden suscribir/desuscribir canales **en tiempo de ejecución**.

## Código *con patrón* (problema resuelto)
La `CuentaObservable` (sujeto) mantiene una lista de **observadores** y los notifica al
cambiar. Cada canal es un observador concreto que se **suscribe** sin que la cuenta
conozca su clase.

In [ ]:
from abc import ABC, abstractmethod


class Observador(ABC):
    @abstractmethod
    def actualizar(self, monto: float, saldo: float) -> None: ...


class NotificadorSMS(Observador):
    def actualizar(self, monto, saldo):
        print(f"   [SMS]   Movimiento de ${monto}. Saldo: ${saldo}")

class NotificadorEmail(Observador):
    def actualizar(self, monto, saldo):
        print(f"   [EMAIL] Movimiento de ${monto}. Saldo: ${saldo}")

class NotificadorApp(Observador):
    def actualizar(self, monto, saldo):
        print(f"   [APP]   Push: movimiento de ${monto}. Saldo: ${saldo}")


class CuentaObservable:
    def __init__(self, saldo: float):
        self.saldo = saldo
        self._observadores: list[Observador] = []

    def suscribir(self, obs: Observador) -> None:
        self._observadores.append(obs)

    def desuscribir(self, obs: Observador) -> None:
        self._observadores.remove(obs)

    def _notificar(self, monto: float) -> None:
        for obs in self._observadores:
            obs.actualizar(monto, self.saldo)

    def registrar_movimiento(self, monto: float) -> None:
        self.saldo += monto
        self._notificar(monto)


cuenta = CuentaObservable(100000)
cuenta.suscribir(NotificadorSMS())
cuenta.suscribir(NotificadorEmail())
app = NotificadorApp()
cuenta.suscribir(app)

print("-- Deposito de 50000 (3 canales) --")
cuenta.registrar_movimiento(50000)

print("-- El cliente desactiva la app; retiro de 20000 --")
cuenta.desuscribir(app)
cuenta.registrar_movimiento(-20000)
print(">> Solucion: los canales se suscriben/desuscriben sin modificar la cuenta.")

-- Deposito de 50000 (3 canales) --
   [SMS]   Movimiento de $50000. Saldo: $150000
   [EMAIL] Movimiento de $50000. Saldo: $150000
   [APP]   Push: movimiento de $50000. Saldo: $150000
-- El cliente desactiva la app; retiro de 20000 --
   [SMS]   Movimiento de $-20000. Saldo: $130000
   [EMAIL] Movimiento de $-20000. Saldo: $130000
>> Solucion: los canales se suscriben/desuscriben sin modificar la cuenta.


### Verificación
- Hay un **sujeto** (`CuentaObservable`) y **3 observadores concretos** (SMS, Email, App).
- Los observadores se **suscriben/desuscriben en tiempo de ejecución**.
- La cuenta notifica sin conocer la clase concreta de cada canal: bajo acoplamiento.

## UML del patrón Observer
```plantuml
@startuml
class CuentaObservable {
    - _observadores : list
    + suscribir(obs)
    + desuscribir(obs)
    + registrar_movimiento(monto)
    + _notificar(monto)
}
interface Observador {
    + actualizar(monto, saldo)
}
Observador <|.. NotificadorSMS
Observador <|.. NotificadorEmail
Observador <|.. NotificadorApp
CuentaObservable o--> Observador : notifica
@enduml
```

## ¿Por qué Observer y no otro patrón?
- El problema es **notificar automáticamente a muchos interesados** cuando cambia el
  estado de la cuenta, sin acoplar el sujeto a ellos. Ese es el caso de Observer.
- No es Strategy: no elegimos **un** algoritmo, sino que informamos a **varios**
  suscriptores a la vez.
- No es Chain of Responsibility: no buscamos que **uno** atienda y detenga la cadena;
  aquí **todos** los suscritos reciben la notificación.
- Observer permite agregar o quitar canales en runtime sin modificar la cuenta.